# 실습 문제 - 101. Tools & Agents

`# TODO` 표시된 부분을 채우며 위에서부터 셀 단위로 실행하세요. (모범답안: `practice_answer_101_Tools_Agents.ipynb`)

### 문제 요약

1. `@tool` 데코레이터로 `calc_discount` 도구 정의 (타입 힌트 + docstring)
2. Pydantic `args_schema`로 `calc_bmi` 도구 정의
3. 두 도구로 에이전트 생성 후 복합 질문으로 도구 호출 확인

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
import os
from langchain.chat_models import init_chat_model

model = init_chat_model(
    os.environ["OPENAI_MODEL"],
    model_provider="openai",
    use_responses_api=True,        # Responses API 사용 (Chat Completions에서는 추론 모델의 도구 호출 불가)
    reasoning={"effort": "low"},   # 추론 강도 낮춤 → 실습 속도·비용 절감
)

## 1. @tool 데코레이터로 calc_discount 도구 정의

타입 힌트가 입력 스키마가 되고, docstring이 모델에게 도구의 용도를 알려줍니다.

In [ ]:
from langchain.tools import tool

# TODO: 아래 함수를 LangChain 도구로 등록하는 데코레이터를 붙이세요.
def calc_discount(price: float, discount_percent: float) -> str:
    """상품의 할인가를 계산합니다.

    Args:
        price: 상품 원가 (원)
        discount_percent: 할인율 (%)
    """
    # TODO: 할인가를 계산하세요. (원가 × (1 - 할인율/100))
    discounted = ...
    return f"원가 {price:,.0f}원의 {discount_percent}% 할인가는 {discounted:,.0f}원입니다."

# 도구 단독 테스트
print(calc_discount.invoke({"price": 50000, "discount_percent": 20}))

## 2. Pydantic 스키마로 calc_bmi 도구 정의

`Field(description=...)`로 각 인자의 의미를 모델에게 명확히 전달합니다.

In [ ]:
from pydantic import BaseModel, Field

class BmiInput(BaseModel):
    """BMI 계산에 사용할 입력 스키마"""
    height_cm: float = Field(description="키 (센티미터 단위)")
    # TODO: weight_kg(몸무게, kg) 필드를 Field(description=...)과 함께 정의하세요.

# TODO: args_schema 에 위에서 정의한 입력 스키마 클래스를 지정하세요.
@tool(args_schema=...)
def calc_bmi(height_cm, weight_kg) -> str:
    """키와 몸무게로 BMI(체질량지수)를 계산합니다."""
    height_m = height_cm / 100
    bmi = weight_kg / (height_m ** 2)
    return f"키 {height_cm}cm, 몸무게 {weight_kg}kg의 BMI는 {bmi:.1f}입니다."

# 도구 단독 테스트
print(calc_bmi.invoke({"height_cm": 175, "weight_kg": 70}))

## 3. 에이전트 생성 + 복합 질문으로 호출

두 도구가 모두 필요한 질문을 던지면, 에이전트가 각 도구를 알맞은 인자로 호출합니다.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    # TODO: 위에서 만든 두 도구를 등록하세요.
    tools=...,
)

result = agent.invoke(
    {"messages": [
        {"role": "system", "content": "당신은 도움이 되는 어시스턴트입니다. 주어진 도구를 이용해 답변하세요."},
        {"role": "user", "content": "키 175cm 몸무게 70kg의 BMI와, 50,000원 상품의 20% 할인가를 알려줘."},
    ]}
)

# 도구 호출 과정 포함 전체 메시지 확인
for msg in result["messages"]:
    msg.pretty_print()

## 확인 포인트

- 출력에서 에이전트가 `calc_bmi`와 `calc_discount`를 **각각 알맞은 인자**로 호출하는지 확인하세요.
- 도구의 docstring과 `Field(description=...)`이 있어야 모델이 어떤 도구를 언제 쓸지 판단할 수 있습니다.